# TED Talks Content Recommender

I wanted a quick recommender that, given a TED talk I liked, suggests a few others I might also like. No collaborative signal here, just TF-IDF on the titles, tags and descriptions, then cosine similarity. The whole thing is small enough to fit in memory.

Data: the Kaggle "TED Talks" dataset by Rounak Banik (`ted_main.csv`, 2,550 talks up to 2017, CC BY-NC-SA 4.0), pulled from a public GitHub copy since Kaggle needs a login. The file also lists TED's own "related talks" for each talk, which gives me a cheap way to check the recommender against something other than my own eyeballs.

In [1]:
import os, json, random
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

random.seed(7)
np.random.seed(7)

DATA = Path('../../data/ted_talks')
DATA.mkdir(parents=True, exist_ok=True)
CSV = DATA / 'ted_talks.csv'

In [2]:
import ast, urllib.request
# public GitHub copy of Kaggle's ted_main.csv (rounakbanik/ted-talks)
URL = "https://raw.githubusercontent.com/amanps/TED-Data-Analysis/master/data/ted_main.csv"
if not CSV.exists():
    urllib.request.urlretrieve(URL, CSV)
    print('downloaded ->', CSV)
else:
    print('using existing', CSV)

downloaded -> ../../data/ted_talks/ted_talks.csv


In [3]:
df = pd.read_csv(CSV)
# tags is a stringified Python list in the source file
df['tags'] = df['tags'].apply(lambda s: ' '.join(t.strip().lower() for t in ast.literal_eval(s)))
df = df[['title', 'main_speaker', 'tags', 'description', 'related_talks', 'views']].reset_index(drop=True)
print(df.shape)
df.head(3)

(2550, 6)


,title,main_speaker,tags,description,related_talks,views
0,Do schools kill creativity?,Ken Robinson,children creativity culture dance education pa...,Sir Ken Robinson makes an entertaining and pro...,"[{'id': 865, 'hero': 'https://pe.tedcdn.com/im...",47227110
1,Averting the climate crisis,Al Gore,alternative energy cars climate change culture...,With the same humor and humanity he exuded in ...,"[{'id': 243, 'hero': 'https://pe.tedcdn.com/im...",3200520
2,Simplicity sells,David Pogue,computers entertainment interface design media...,New York Times columnist David Pogue takes aim...,"[{'id': 1725, 'hero': 'https://pe.tedcdn.com/i...",1636292


## Build the TF-IDF feature matrix

I'm concatenating title, tags, and description into a single text blob per talk. Tags get repeated twice on purpose so the vectorizer weights them a bit more than prose.

In [4]:
df['blob'] = (df['title'].fillna('') + ' ' +
              (df['tags'].fillna('') + ' ') * 2 +
              df['description'].fillna(''))

vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, max_df=0.8)
X = vec.fit_transform(df['blob'])
print('tfidf shape:', X.shape, ' vocab size:', len(vec.vocabulary_))

tfidf shape: (2550, 14677)  vocab size: 14677


In [5]:
sim = cosine_similarity(X)
# diagonal is 1.0 by definition; zero it out so a talk doesn't recommend itself
np.fill_diagonal(sim, 0.0)
sim.shape

(2550, 2550)

## Recommend

Given the index of a seed talk, return the top-k most similar. I added a small diversity tweak — if two candidates share the exact same tag string I skip the second. Without it, talks that share the exact same tag list can crowd the top-5.

In [6]:
def recommend(seed_idx, k=5, dedup_tags=True):
    scores = sim[seed_idx]
    order = np.argsort(-scores)
    out, seen_tags = [], set()
    for j in order:
        if dedup_tags and df.loc[j, 'tags'] in seen_tags:
            continue
        seen_tags.add(df.loc[j, 'tags'])
        out.append((j, float(scores[j])))
        if len(out) == k:
            break
    return out

seed = 0
print('SEED ->', df.loc[seed, 'title'], '|', df.loc[seed, 'tags'])
print()
for j, s in recommend(seed, k=5):
    print(f'  {s:.3f}  {df.loc[j, "title"]}')

SEED -> Do schools kill creativity? | children creativity culture dance education parenting teaching

  0.323  Bring on the learning revolution!
  0.245  Kids need structure
  0.234  How to escape education's death valley
  0.226  What adults can learn from kids
  0.213  Dancing with light


In [7]:
# offline check against TED's own editorial "related talks" (usually 3-6 per talk)
title_to_idx = {t: i for i, t in enumerate(df['title'])}
hits, has_related, recall = 0, 0, []
for i in range(len(df)):
    related = {title_to_idx[r['title']] for r in ast.literal_eval(df.loc[i, 'related_talks'])
               if r['title'] in title_to_idx}
    if not related:
        continue
    has_related += 1
    recs = {j for j, _ in recommend(i, k=10, dedup_tags=False)}
    found = len(recs & related)
    hits += found > 0
    recall.append(found / len(related))

# a random recommender for scale
rng = np.random.default_rng(7)
rand_recall = []
for i in range(len(df)):
    related = {title_to_idx[r['title']] for r in ast.literal_eval(df.loc[i, 'related_talks']) if r['title'] in title_to_idx}
    if related:
        recs = set(rng.choice(len(df), 10, replace=False))
        rand_recall.append(len(recs & related) / len(related))

print(f"talks with related talks in the file: {has_related}")
print(f"TF-IDF top-10: hit rate {hits / has_related:.3f}, mean recall {np.mean(recall):.3f}")
print(f"random top-10: mean recall {np.mean(rand_recall):.3f}")

talks with related talks in the file: 2550
TF-IDF top-10: hit rate 0.771, mean recall 0.271
random top-10: mean recall 0.003


## Notes

- The seed check reads well. For Ken Robinson's "Do schools kill creativity?" the top hits are his own follow-up ("Bring on the learning revolution!", and "How to escape education's death valley") plus other talks on kids and learning. The fifth one, "Dancing with light", is there because of the "dance" tag, which shows how literal tag matching is.
- Against TED's own editorial "related talks" list, the TF-IDF top-10 contains at least one of them for 77.1% of talks, and recovers 27.1% of the related talks on average. A random top-10 recovers 0.3%. TED's lists aren't ground truth for what a viewer would like, but they're a useful independent check.
- Cosine similarity on TF-IDF is a baseline. Sentence embeddings (e.g. MiniLM) would probably do better at the description level, and the related-talks check above gives an easy way to measure that.
- The cold-start problem doesn't really exist for content-based methods (every new talk is recommendable as soon as it's in the index), but the flip side is that it can't pick up things only humans would notice, like a great speaker or a memorable Q&A. The file also has view counts, which could be blended in as a popularity prior.
- I'd swap the dedup-by-tags hack for MMR if I were taking this further.